In [2]:
import pandas as pd
import numpy as np

sol = pd.read_csv("FinAcceso_solicitudes_credito.csv")
mer = pd.read_csv("FinAcceso_mercado_potencial.csv")

### Punto 3

In [3]:
sol["decision"].value_counts().reset_index()

,decision,count
0,Rechazada,662
1,Aprobada,658


In [3]:
sol["aprobado"] = (sol["decision"] == "Aprobada").astype(int)
sol["aprobado"].mean() * 100


np.float64(49.848484848484844)

La tasa global de aprobación es 49,8 %

In [34]:
t3 = sol.groupby("zona").agg(
    solicitudes=("aprobado", "count"),
    aprobaciones=("aprobado", "sum"),
    tasa_aprob=("aprobado", "mean")
)
t3["tasa_aprob"] = (t3["tasa_aprob"] * 100)
t3.sort_values("aprobaciones", ascending=False)

,solicitudes,aprobaciones,tasa_aprob
zona,,,
Norte,475,295,62.105263
Centro,395,193,48.860759
Sur,318,129,40.566038
Periferia,132,41,31.060606


-El orden de las zonas no cambia entre números absolutos y tasas

-Sería engañoso interpretar que Norte tiene más demanda o que allí la asesoría rinde más. Ni las aprobaciones ni la tasa miden necesidad la tasa refleja quienes lo solicitan.

-Antes de comparar tasas entre zonas conviene revisar el puntaje de riesgo, porque si los solicitantes de Periferia tienen puntajes más bajos, su menor tasa se explicaría por el puntaje y no por la zona, y ese puntaje heredado podría reflejar desigualdades previas.

### Punto 4


In [30]:
sol.groupby("zona")["puntaje_riesgo"].median()

zona
Centro       659.0
Norte        670.0
Periferia    594.0
Sur          614.5
Name: puntaje_riesgo, dtype: float64

In [31]:
sol["historial_crediticio"].value_counts()

historial_crediticio
Amplio      824
Limitado    496
Name: count, dtype: int64

In [32]:
def rango_puntaje(p):
    if p < 610:
        return "<610"
    elif p < 690:
        return "610-689"
    else:
        return "690+"

sol["rango_puntaje"] = sol["puntaje_riesgo"].apply(rango_puntaje)
sol["rango_puntaje"].value_counts()

rango_puntaje
610-689    490
<610       448
690+       382
Name: count, dtype: int64

In [33]:
t4 = sol.groupby(["rango_puntaje", "historial_crediticio"]).agg(
    solicitudes=("decision", "count"),
    pct_aprob=("decision", lambda x: (x == "Aprobada").mean() * 100)
)
t4

solicitudes  pct_aprob
rango_puntaje historial_crediticio                        
610-689       Amplio                        319  58.307210
              Limitado                      171  44.444444
690+          Amplio                        318  82.704403
              Limitado                       64  73.437500
<610          Amplio                        187  24.598930
              Limitado                      261  15.325670

En todos los intervalos de puntaje, las personas con historial amplio se aprueban más que las de historial limitado, y la mayoría de quienes tienen historial limitado están en el rango de puntaje más bajo.

No se puede concluir.Faltaría revisar ingreso, carga financiera, monto, canal y zona, que podrían explicar parte de la brecha. Tampoco sabemos cómo se construyó el puntaje heredado de puntaje de riesgo, si ya incluye el historial, el limitado se penaliza dos veces. Y no tenemos datos de mora ni de quienes nunca solicitaron, así que no sabemos si el historial limitado realmente implica más riesgo.